## PHASE 1

* Load Multi-XScience via Hugging Face, inspect structure, clean text (remove citation markers like @cite_1, extra whitespace/artifacts) 
* Organize into a clean input papers (cluster) → target summary format we'll reuse in every later phase.

In [16]:
import os

for root, dirs, files in os.walk("Multi-XScience"):
    for f in files:
        print(os.path.join(root, f))

Multi-XScience\LICENSE
Multi-XScience\README.md
Multi-XScience\.git\config
Multi-XScience\.git\description
Multi-XScience\.git\HEAD
Multi-XScience\.git\index
Multi-XScience\.git\packed-refs
Multi-XScience\.git\hooks\applypatch-msg.sample
Multi-XScience\.git\hooks\commit-msg.sample
Multi-XScience\.git\hooks\fsmonitor-watchman.sample
Multi-XScience\.git\hooks\post-update.sample
Multi-XScience\.git\hooks\pre-applypatch.sample
Multi-XScience\.git\hooks\pre-commit.sample
Multi-XScience\.git\hooks\pre-merge-commit.sample
Multi-XScience\.git\hooks\pre-push.sample
Multi-XScience\.git\hooks\pre-rebase.sample
Multi-XScience\.git\hooks\pre-receive.sample
Multi-XScience\.git\hooks\prepare-commit-msg.sample
Multi-XScience\.git\hooks\push-to-checkout.sample
Multi-XScience\.git\hooks\sendemail-validate.sample
Multi-XScience\.git\hooks\update.sample
Multi-XScience\.git\info\exclude
Multi-XScience\.git\logs\HEAD
Multi-XScience\.git\logs\refs\heads\master
Multi-XScience\.git\logs\refs\remotes\origin\HEA

In [17]:
import gzip
import json

def load_json_gz(path):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        data = json.load(f) 
    return data

train_data = load_json_gz("Multi-XScience/data/train.json.gz")
val_data = load_json_gz("Multi-XScience/data/val.json.gz")
test_data = load_json_gz("Multi-XScience/data/test.json.gz")

print("Train size:", len(train_data))
print("Val size:", len(val_data))
print("Test size:", len(test_data))

Train size: 30369
Val size: 5066
Test size: 5093


In [18]:
example = train_data[0]

print("TARGET PAPER ABSTRACT:\n", example["abstract"])
print("\nRELATED WORK (gold summary):\n", example["related_work"])
print("\nCITED PAPERS (input cluster):")
for cite_key, cite_data in example["ref_abstract"].items():
    print(f"\n[{cite_key}]:", cite_data["abstract"][:200], "...")

TARGET PAPER ABSTRACT:
 Author(s): Kuperberg, Greg; Thurston, Dylan P. | Abstract: We give a purely topological definition of the perturbative quantum invariants of links and 3-manifolds associated with Chern-Simons field theory. Our definition is as close as possible to one given by Kontsevich. We will also establish some basic properties of these invariants, in particular that they are universally finite type with respect to algebraically split surgery and with respect to Torelli surgery. Torelli surgery is a mutual generalization of blink surgery of Garoufalidis and Levine and clasper surgery of Habiro.

RELATED WORK (gold summary):
 Two other generalizations that can be considered are invariants of graphs in 3-manifolds, and invariants associated to other flat connections @cite_16 . We will analyze these in future work. Among other things, there should be a general relation between flat bundles and links in 3-manifolds on the one hand and finite covers and branched covers on the ot

In [19]:
import re

def clean_target_abstract(text):
    """Remove the 'Author(s): ... | Abstract: ...' prefix, keep just the abstract."""
    if "Abstract:" in text:
        text = text.split("Abstract:", 1)[1]
    return text.strip()

def clean_related_work(text):
    """Remove @cite_N markers from the gold summary for baseline training."""
    text = re.sub(r"\s*@cite_\d+\s*", " ", text)  
    text = re.sub(r"\s+([.,])", r"\1", text)       
    text = re.sub(r"\s+", " ", text)            
    return text.strip()

def get_cited_abstracts(ref_abstract_dict):
    """Flatten the ref_abstract dict into a simple list of cited paper abstracts."""
    return [v["abstract"].strip() for v in ref_abstract_dict.values() if v.get("abstract")]

def preprocess_example(example):
    return {
        "target_abstract": clean_target_abstract(example["abstract"]),
        "cited_abstracts": get_cited_abstracts(example["ref_abstract"]),
        "gold_summary": clean_related_work(example["related_work"]),
    }

In [20]:
processed = preprocess_example(train_data[0])

print("TARGET ABSTRACT:\n", processed["target_abstract"])
print("\nCITED ABSTRACTS (input cluster):")
for i, abs_text in enumerate(processed["cited_abstracts"]):
    print(f"[{i}]:", abs_text[:150], "...")
print("\nGOLD SUMMARY (cleaned):\n", processed["gold_summary"])

TARGET ABSTRACT:
 We give a purely topological definition of the perturbative quantum invariants of links and 3-manifolds associated with Chern-Simons field theory. Our definition is as close as possible to one given by Kontsevich. We will also establish some basic properties of these invariants, in particular that they are universally finite type with respect to algebraically split surgery and with respect to Torelli surgery. Torelli surgery is a mutual generalization of blink surgery of Garoufalidis and Levine and clasper surgery of Habiro.

CITED ABSTRACTS (input cluster):
[0]: This note is a sequel to our earlier paper of the same title [4] and describes invariants of rational homology 3-spheres associated to acyclic orthogo ...
[1]: Recently, Mullins calculated the Casson-Walker invariant of the 2-fold cyclic branched cover of an oriented link in S^3 in terms of its Jones polynomi ...

GOLD SUMMARY (cleaned):
 Two other generalizations that can be considered are invariants of grap

In [21]:
def process_and_save(data, split_name):
    processed = [preprocess_example(ex) for ex in data]
    with open(f"processed_{split_name}.json", "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=2)
    print(f"{split_name}: {len(processed)} examples saved.")

process_and_save(train_data, "train")
process_and_save(val_data, "val")
process_and_save(test_data, "test")

train: 30369 examples saved.
val: 5066 examples saved.
test: 5093 examples saved.


In [22]:
import json

def load_processed(split_name):
    with open(f"processed_{split_name}.json", "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_processed("train")
val_data = load_processed("val")
test_data = load_processed("test")

print(len(train_data), len(val_data), len(test_data))
print(train_data[0].keys())  # should show: dict_keys(['target_abstract', 'cited_abstracts', 'gold_summary'])

30369 5066 5093
dict_keys(['target_abstract', 'cited_abstracts', 'gold_summary'])


## PHASE 2

In [23]:
def format_input(example):
    # Join all cited abstracts with a separator
    input_text = " </s> ".join(example["cited_abstracts"])
    return input_text

# Quick check
print(format_input(train_data[0])[:500])

This note is a sequel to our earlier paper of the same title [4] and describes invariants of rational homology 3-spheres associated to acyclic orthogonal local systems. Our work is in the spirit of the Axelrod–Singer papers [1], generalizes some of their results, and furnishes a new setting for the purely topological implications of their work. </s> Recently, Mullins calculated the Casson-Walker invariant of the 2-fold cyclic branched cover of an oriented link in S^3 in terms of its Jones polyno


In [24]:
from datasets import Dataset

train_subset = train_data[:3000]
val_subset = val_data[:300]   # keep val small too, just for tracking loss during training

def build_hf_dataset(data):
    return Dataset.from_dict({
        "input_text": [format_input(ex) for ex in data],
        "target_text": [ex["gold_summary"] for ex in data],
    })

train_ds = build_hf_dataset(train_subset)
val_ds = build_hf_dataset(val_subset)

print(train_ds)
print(train_ds[0]["input_text"][:300])
print(train_ds[0]["target_text"])

Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 3000
})
This note is a sequel to our earlier paper of the same title [4] and describes invariants of rational homology 3-spheres associated to acyclic orthogonal local systems. Our work is in the spirit of the Axelrod–Singer papers [1], generalizes some of their results, and furnishes a new setting for the 
Two other generalizations that can be considered are invariants of graphs in 3-manifolds, and invariants associated to other flat connections. We will analyze these in future work. Among other things, there should be a general relation between flat bundles and links in 3-manifolds on the one hand and finite covers and branched covers on the other hand.


In [25]:
from transformers import BartTokenizerFast, BartForConditionalGeneration

model_name = "facebook/bart-base"
tokenizer = BartTokenizerFast.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 128

def tokenize_fn(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding="max_length",
    )
    labels = tokenizer(
        batch["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_ds.map(tokenize_fn, batched=True, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(tokenize_fn, batched=True, remove_columns=val_ds.column_names)

Map: 100%|██████████| 300/300 [00:00<00:00, 1579.16 examples/s]


In [26]:
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Using device:", device)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),  # speeds up training on GPU
    logging_steps=50,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,   
    data_collator=data_collator,
)

Using device: cuda


In [27]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.320890,3.115288
2,3.116812,3.069513
3,3.025162,3.050426


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.84it/s]


TrainOutput(global_step=2250, training_loss=3.235066413031684, metrics={'train_runtime': 918.1811, 'train_samples_per_second': 9.802, 'train_steps_per_second': 2.45, 'total_flos': 2743814062080000.0, 'train_loss': 3.235066413031684, 'epoch': 3.0})

Evaluating baseline model (fine-tuned BART).

In [28]:
def generate_summary(input_text, max_length=128):
    inputs = tokenizer(
        input_text, return_tensors="pt", max_length=MAX_INPUT_LEN, truncation=True
    ).to(device)
    
    output_ids = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=4,
        early_stopping=True,
    )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

for i in range(3):
    example = val_subset[i]
    input_text = format_input(example)
    generated = generate_summary(input_text)
    
    print(f"--- Example {i} ---")
    print("GOLD SUMMARY:\n", example["gold_summary"])
    print("\nGENERATED SUMMARY:\n", generated)
    print("\n" + "="*80 + "\n")

--- Example 0 ---
GOLD SUMMARY:
 In his discussion of functional testing, Howden stresses the need to identify input domains, and gives guidelines for systematic selection of test points for several types of input values that occur in scientific programs. Our basic philosophy is similar to Howden's; we develop this point of view further, by automating part of the selection process.

GENERATED SUMMARY:
 The most reliable testing technique for discovering errors in a program is structural testing. The method is based on structural testing, where the test data is defined as an integrated collection of functions. The selection of test data depends on the functions used in the design and on the value spaces over which the functions are defined. Structural testing is a more general approach to functional testing.


--- Example 1 ---
GOLD SUMMARY:
 Gourlay presents a precise framework for the discussion of issues in testing. In his terminology, our test selection criteria are a special form o

In [29]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore

test_subset = test_data[:300]

generated_summaries = []
gold_summaries = []

for ex in test_subset:
    input_text = format_input(ex)
    generated = generate_summary(input_text)
    generated_summaries.append(generated)
    gold_summaries.append(ex["gold_summary"])

print("Generated", len(generated_summaries), "summaries.")

Generated 300 summaries.


In [31]:
scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=True)

rouge1_scores, rouge2_scores, rougeL_scores = [], [], []


for gen, gold in zip(generated_summaries, gold_summaries):
    scores = scorer.score(gold, gen)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)


print("Average ROUGE-1 Score: ", sum(rouge1_scores)/len(rouge1_scores))
print("Average ROUGE-2 Score: ", sum(rouge2_scores)/len(rouge2_scores))
print("Average ROUGe-L Score: ", sum(rougeL_scores)/len(rougeL_scores))

from bert_score import score as bertscore 

P, R, F1 = bertscore(generated_summaries, gold_summaries, lang="en", verbose=True)

print(f"BERT Metrics: Precision: {P}, Recall: {R}, F1-Score: {F1}")
print("Average BERTScore F1:", F1.mean().item())

Average ROUGE-1 Score:  0.2801565230942524
Average ROUGE-2 Score:  0.049279449601409515
Average ROUGe-L Score:  0.16092846049452322


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 4904.69it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


100%|██████████| 10/10 [00:27<00:00,  2.78s/it]


computing greedy matching.


100%|██████████| 5/5 [00:00<00:00, 27.52it/s]

done in 28.01 seconds, 10.71 sentences/sec
BERT Metrics: Precision: tensor([0.8772, 0.8413, 0.8492, 0.8507, 0.8692, 0.7919, 0.8290, 0.8427, 0.8756,
        0.8415, 0.8441, 0.8334, 0.8399, 0.8664, 0.8360, 0.8541, 0.8654, 0.8805,
        0.8686, 0.8493, 0.8559, 0.8639, 0.8603, 0.8654, 0.8281, 0.8458, 0.8667,
        0.8480, 0.8653, 0.8442, 0.8621, 0.8541, 0.8499, 0.8393, 0.8525, 0.8754,
        0.8502, 0.8600, 0.8486, 0.8695, 0.8506, 0.8517, 0.8429, 0.8406, 0.8663,
        0.8721, 0.8292, 0.8576, 0.8681, 0.8643, 0.8309, 0.8670, 0.8547, 0.9076,
        0.8434, 0.8712, 0.8562, 0.8971, 0.8664, 0.8226, 0.8564, 0.8679, 0.8522,
        0.8444, 0.8526, 0.8490, 0.8247, 0.8476, 0.8334, 0.8469, 0.8531, 0.8212,
        0.8414, 0.8694, 0.8578, 0.8561, 0.8580, 0.8697, 0.8456, 0.8686, 0.8571,
        0.8696, 0.8519, 0.8475, 0.8471, 0.8456, 0.8647, 0.8658, 0.8708, 0.8531,
        0.8524, 0.8444, 0.8516, 0.8564, 0.8578, 0.8496, 0.8665, 0.8578, 0.8520,
        0.8315, 0.8434, 0.8621, 0.8627, 0.8628, 0.83

These are reasonable, legitimate baseline numbers — not spectacular, but that's expected and actually good for your project story. A weak-ish baseline gives your later improvements (section-awareness, redundancy reduction) real room to show measurable gains.

ROUGE-2 being notably lower than ROUGE-1 (0.049 vs 0.280) is normal — ROUGE-2 checks exact 2-word phrase overlap, which is much stricter. This just confirms the model is paraphrasing rather than copying phrases verbatim — expected for an abstractive model.

BERTScore (0.848) is meaningfully higher than ROUGE scores would suggest — this tells you the model is capturing the right meaning even when using different words than the gold summary. This is a good sign, and also a nice talking point: "ROUGE alone would undersell this model; BERTScore shows it's semantically closer than raw word-overlap suggests."

In [32]:
baseline_results = {
    "model": "bart-base",
    "train_size": 3000,
    "epochs": 3,
    "rouge1": 0.2801565230942524,
    "rouge2": 0.049279449601409515,
    "rougeL": 0.16092846049452322,
    "bertscore_f1": 0.8476278781890869,
}

with open("baseline_results.json", "w") as f:
    json.dump(baseline_results, f, indent=2)

print("Saved baseline results.")

Saved baseline results.


In [33]:
model.save_pretrained("./bart_baseline_model")
tokenizer.save_pretrained("./bart_baseline_model")
print("Model saved.")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.81it/s]

Model saved.
